In [1]:
%load_ext rpy2.ipython

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append(str(Path("../src").resolve()))

from river_flux.pipeline import run_station_parameter

In [2]:
# %%R
# install.packages("EGRET", repos="https://cloud.r-project.org")

# Estimate fluxes

Code to estimate annual fluxes using four different methods:

 1. OSPAR
 2. EGRET
 3. EGRET Flow-Normalised
 4. GAM (with 95 % CI)

In [3]:
pars = [
    "TOTN_ug/l",
    "NO3-N_ug/l",
    "TOTP_ug/l",
    "PO4-P_ug/l",
    "TOC_mg/l",
    "SPM_mg/l",
]
stns = [
    "BUSEDRA",
    "TELESKI",
    "VESENUM",
    "ØSTEGLO",
]

# File paths
chemistry_file = "../data/oslofjord_rid_chem.xlsx"
flow_file = "../data/oslofjord_rid_flows.xlsx"
wrtds_script = "../scripts/run_egret.R"
gam_script = "../scripts/run_gam.R"
output_dir = Path("../output")
plot_dir = output_dir / "plots"

In [4]:
# Create output dirs if needed
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)
plot_dir.mkdir(
    parents=True,
    exist_ok=True,
)

# Containers for results
all_fluxes = []
all_trends = []

# Process stations and pars
for stn in stns:
    for par in pars:
        par_name = par.split("_")[0]
        print(f"Processing {stn} | {par}")
        try:
            result = run_station_parameter(
                station=stn,
                parameter=par,
                chemistry_file=chemistry_file,
                flow_file=flow_file,
                wrtds_script=wrtds_script,
                gam_script=gam_script,
            )

            # Fluxes
            flux_df = result["data"].copy().reset_index()
            cols = [col.replace(f"_{par_name}", "") for col in flux_df.columns]
            flux_df.columns = cols
            flux_df["station"] = stn
            flux_df["parameter"] = par_name
            flux_df["units"] = "tonnes"
            all_fluxes.append(flux_df)

            # Trends
            trend_df = result["trends"].copy().reset_index()
            trend_df["station"] = stn
            trend_df["parameter"] = par_name
            trend_df["units"] = "tonnes"
            all_trends.append(trend_df)

            # Plot
            fig = result["figure"]
            plot_file = plot_dir / f"{stn}_{par_name}_tonnes.png"
            fig.savefig(
                plot_file,
                dpi=200,
                bbox_inches="tight",
            )
            plt.close(fig)

            print(f"  Saved {plot_file}")

        except Exception as e:
            print(f"FAILED: {stn} | {par}")
            print(e)

# Combine results
fluxes_df = pd.concat(
    all_fluxes,
    ignore_index=True,
    axis="rows",
)

trends_df = pd.concat(
    all_trends,
    ignore_index=True,
    axis="rows",
)

# Tidy fluxes
cols = [
    "station",
    "parameter",
    "units",
    "year",
    "OSPAR_tonn",
    "EGRET_tonn",
    "EGRET_FN_tonn",
    "GAM_tonn",
    "lower_GAM_tonn",
    "upper_GAM_tonn",
]
fluxes_df = fluxes_df[cols]
for col in fluxes_df.columns:
    if col.endswith("_tonn"):
        fluxes_df[col] = fluxes_df[col].round(-1)

# Tidy trends
cols = [
    "station",
    "parameter",
    "units",
    "method",
    "trend",
    "p_value",
    "slope_tonnes-per-year",
    "intercept",
    "pct_change",
]
trends_df = trends_df[cols]
trends_df["p_value"] = trends_df["p_value"].round(6)
trends_df["slope_tonnes-per-year"] = trends_df["slope_tonnes-per-year"].round(1)
trends_df["intercept"] = trends_df["intercept"].round(1)
trends_df["pct_change"] = trends_df["pct_change"].round(1)

# Save
fluxes_file = output_dir / "annual_fluxes_tonnes.csv"
trends_file = output_dir / "trend_results_tonnes.csv"
fluxes_df.to_csv(
    fluxes_file,
    index=False,
)
trends_df.to_csv(
    trends_file,
    index=False,
)

print(f"Saved {fluxes_file}")
print(f"Saved {trends_file}")
print(f"Saved plots to {plot_dir}")

Processing BUSEDRA | TOTN_ug/l
Adding flow offset of 47.848315 m3/s to avoid zero flows.
Reading input files...
Building eList...
Running WRTDS model...

 Discharge Record is 12784 days long, which is 35 years
 First day of the discharge record is 1990-01-01 and last day is 2024-12-31
 The water quality record has 579 samples
 The first sample is from 1990-01-11 and the last sample is from 2024-12-02
 Discharge: Minimum, mean and maximum 47.8 385 1700
 Concentration: Minimum, mean and maximum 0.21 0.43 1.2
 Percentage of the sample values that are censored is 0 %
 first step running estCrossVal may take about 1 minute
 estCrossVal % complete:
0 	1 	2 	3 	4 	5 	6 	7 	8 	9 	10 	
11 	12 	13 	14 	15 	16 	17 	18 	19 	20 	
21 	22 	23 	24 	25 	26 	27 	28 	29 	30 	
31 	32 	33 	34 	35 	36 	37 	38 	39 	40 	
41 	42 	43 	44 	45 	46 	47 	48 	49 	50 	
51 	52 	53 	54 	55 	56 	57 	58 	59 	60 	
61 	62 	63 	64 	65 	66 	67 	68 	69 	70 	
71 	72 	73 	74 	75 	76 	77 	78 	79 	80 	
81 	82 	83 	84 	85 	86 	87 